# LeetCode 1356 — Sort Integers by The Number of 1 Bits bằng Python

**Link đề:** https://leetcode.com/problems/sort-integers-by-the-number-of-1-bits/
**Độ khó:** Easy · **Chủ đề:** Array, Bit Manipulation, Sorting, Counting

```python
class Solution:
    def sortByBits(self, arr: list[int]) -> list[int]:
        ...
```

### Mục tiêu học
1. Hiểu biểu diễn **nhị phân** và cách đếm số bit `1` (popcount) bằng nhiều cách: `bin().count`, vòng `& 1` / `>>`, mẹo `n & (n - 1)`, `int.bit_count()`.
2. Nắm chắc `sorted(..., key=...)` và **khóa dạng tuple** `(tiêu chí 1, tiêu chí 2)` để sắp theo nhiều tiêu chí.
3. So sánh 4 lời giải: key tuple → comparator `cmp_to_key` → mã hoá khóa thành một số → **counting/bucket** theo số bit.
4. Trace từng bước, chứng minh, phân tích Time/Space, và áp dụng "sắp theo nhiều tiêu chí" vào biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho mảng số nguyên `arr`. Sắp xếp các số **tăng dần theo số bit 1** trong biểu diễn nhị phân. Nếu hai số có **cùng** số bit 1 thì sắp **tăng dần theo giá trị**. Trả về mảng đã sắp.

| Input | Output |
|---|---|
| `[0,1,2,3,4,5,6,7,8]` | `[0,1,2,4,8,3,5,6,7]` |
| `[1024,512,256,128,64,32,16,8,4,2,1]` | `[1,2,4,8,16,32,64,128,256,512,1024]` |

Giải thích ví dụ 1: `0` có 0 bit; `1,2,4,8` có 1 bit; `3,5,6` có 2 bit; `7` có 3 bit.

**Constraints:** `1 <= arr.length <= 500`, `0 <= arr[i] <= 10^4`.

Câu hỏi đặt ra:
- Làm sao đếm số bit 1 của một số?
- Làm sao bảo Python "sắp theo tiêu chí A, hòa thì theo tiêu chí B"?

In [1]:
arr1 = [0, 1, 2, 3, 4, 5, 6, 7, 8]
arr2 = [1024, 512, 256, 128, 64, 32, 16, 8, 4, 2, 1]
for x in arr1:
    print(x, "→", bin(x))

0 → 0b0
1 → 0b1
2 → 0b10
3 → 0b11
4 → 0b100
5 → 0b101
6 → 0b110
7 → 0b111
8 → 0b1000


## 2. Nền tảng: hệ nhị phân và `bin()`

Hệ nhị phân chỉ dùng chữ số `0` và `1`; mỗi vị trí mạnh gấp **2** lần vị trí bên phải. Ví dụ `6 = 110₂` vì `1·4 + 1·2 + 0·1 = 6`.

`bin(x)` trả về **chuỗi** có tiền tố `'0b'`: `bin(6) == '0b110'`. Bit 1 của 6 là 2 bit.

In [2]:
x = 6
print(bin(x))            # '0b110'
print(bin(x)[2:])        # bỏ '0b' bằng slicing → '110'
value = 0
for bit in "110":
    value = value * 2 + int(bit)   # đọc nhị phân như đọc số thập phân
print("Đọc lại:", value)

0b110
110
Đọc lại: 6


## 3. Bốn cách đếm số bit 1 (popcount)

### 3.1. Đếm ký tự `'1'` trong chuỗi
`str.count(sub)` đếm số lần chuỗi con `sub` xuất hiện. `bin(x).count('1')` — tiền tố `'0b'` không chứa `'1'` nên không ảnh hưởng.

In [3]:
def popcount_str(x):
    return bin(x).count("1")

for x in [0, 1, 6, 7, 1024, 10000]:
    print(x, bin(x), popcount_str(x))

0 0b0 0
1 0b1 1
6 0b110 2
7 0b111 3
1024 0b10000000000 1
10000 0b10011100010000 5


### 3.2. Duyệt từng bit bằng `& 1` và `>> 1`

- `x & 1` lấy **bit thấp nhất** (0 hoặc 1). `&` là AND từng bit: bit kết quả là 1 khi cả hai bit đều 1.
- `x >> 1` **dịch phải** 1 bit = bỏ bit thấp nhất (giống `x // 2`).

Lặp đến khi `x == 0`. Số vòng = số bit của `x` (với `x ≤ 10^4` là ≤ 14 vòng).

In [4]:
print("6 & 1 =", 6 & 1, "| 7 & 1 =", 7 & 1)
print("6 >> 1 =", 6 >> 1, "(110 → 11)")

def popcount_shift(x):
    count = 0
    while x:
        count += x & 1
        x >>= 1
    return count

print([popcount_shift(x) for x in [0, 1, 6, 7, 1024, 10000]])

6 & 1 = 0 | 7 & 1 = 1
6 >> 1 = 3 (110 → 11)
[0, 1, 2, 3, 1, 5]


### 3.3. Mẹo Brian Kernighan: `x & (x - 1)` xoá bit 1 thấp nhất

Trừ 1 làm bit 1 thấp nhất thành 0 và các bit 0 phía sau thành 1: `12 = 1100`, `11 = 1011`, `12 & 11 = 1000`. Mỗi vòng xoá đúng một bit 1 → số vòng = số bit 1.

In [5]:
x = 12
print(f"{x:>2} = {x:04b}")
print(f"{x-1:>2} = {x-1:04b}")
print(f"&  = {x & (x-1):04b}")

def popcount_kernighan(x):
    count = 0
    while x:
        x &= x - 1
        count += 1
    return count

print([popcount_kernighan(x) for x in [0, 1, 6, 7, 1024, 10000]])

12 = 1100
11 = 1011
&  = 1000
[0, 1, 2, 3, 1, 5]


`f"{x:04b}"` là **định dạng f-string**: `b` = in nhị phân, `04` = rộng 4 ký tự, đệm `0` bên trái.

### 3.4. `int.bit_count()` (Python 3.10+)

Python có sẵn phương thức `x.bit_count()`. LeetCode dùng Python mới nên có thể dùng; ở đây ta viết có dự phòng cho máy cũ.

In [6]:
def popcount_builtin(x):
    if hasattr(int, "bit_count"):
        return x.bit_count()
    return bin(x).count("1")

assert all(popcount_str(x) == popcount_shift(x) == popcount_kernighan(x) == popcount_builtin(x)
           for x in range(0, 10001))
print("4 cách popcount khớp nhau với mọi x trong [0, 10000]")

4 cách popcount khớp nhau với mọi x trong [0, 10000]


## 4. Nền tảng: `sorted` và tham số `key`

- `sorted(a)` trả về **list mới** đã sắp tăng dần; `a.sort()` sắp **tại chỗ** và trả `None`.
- `key=f`: Python tính `f(x)` cho mỗi phần tử rồi **so sánh các giá trị `f(x)`** thay vì so sánh `x`.

In [7]:
words = ["banana", "kiwi", "apple", "fig"]
print(sorted(words))               # theo bảng chữ cái
print(sorted(words, key=len))      # theo độ dài
a = [3, 1, 2]
print(a.sort(), a)                 # sort() trả None, a đã bị đổi

['apple', 'banana', 'fig', 'kiwi']
['fig', 'kiwi', 'apple', 'banana']
None [1, 2, 3]


### 4.1. So sánh **tuple** — chìa khoá của bài

Python so sánh tuple **từ trái sang phải**: so phần tử đầu; nếu bằng nhau mới so phần tử thứ hai,… (giống tra từ điển). Vậy `key=lambda x: (A(x), B(x))` nghĩa là "sắp theo A, hòa thì theo B".

`lambda x: ...` là cách viết **hàm vô danh** ngắn gọn: `lambda x: x * 2` tương đương `def f(x): return x * 2`.

In [8]:
print((1, 8) < (2, 3))    # True: 1 < 2, không cần xem phần sau
print((2, 3) < (2, 5))    # True: hòa phần đầu, 3 < 5
print((2, 5) < (2, 5))    # False: bằng nhau

double = lambda x: x * 2
print(double(21))

True
True
False
42


In [9]:
for x in [3, 4, 5, 8]:
    print(x, "→ khóa", (popcount_str(x), x))

3 → khóa (2, 3)
4 → khóa (1, 4)
5 → khóa (2, 5)
8 → khóa (1, 8)


### 4.2. Tính ổn định (stable) của `sorted`

`sorted` của Python là **stable**: các phần tử có khóa bằng nhau giữ nguyên thứ tự ban đầu. Mẹo: sắp theo tiêu chí phụ trước, rồi sắp theo tiêu chí chính — kết quả giống key tuple.

In [10]:
a = [7, 3, 8, 5, 4, 6]
step1 = sorted(a)                          # tiêu chí phụ: giá trị
step2 = sorted(step1, key=popcount_str)    # tiêu chí chính: số bit (stable)
print(step1)
print(step2)
print(sorted(a, key=lambda x: (popcount_str(x), x)))

[3, 4, 5, 6, 7, 8]
[4, 8, 3, 5, 6, 7]
[4, 8, 3, 5, 6, 7]


## 5. Cách nghĩ trực giác (baseline): tự tính khóa rồi sắp cặp

Cách người mới dễ nghĩ: tạo list các cặp `(số bit, giá trị)`, sắp list cặp, rồi lấy lại giá trị. Đây chính là ý tưởng "decorate – sort – undecorate".

In [11]:
def sort_by_bits_pairs(arr):
    pairs = []
    for x in arr:
        pairs.append((popcount_shift(x), x))   # decorate
    pairs.sort()                                # sort theo tuple
    return [x for _, x in pairs]                # undecorate

print(sort_by_bits_pairs(arr1))
print(sort_by_bits_pairs(arr2))

[0, 1, 2, 4, 8, 3, 5, 6, 7]
[1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]


`for _, x in pairs`: **unpacking** tuple 2 phần tử; `_` là tên quy ước cho giá trị "không dùng tới".

Baseline này đã đúng và đủ nhanh. Câu hỏi tiếp theo: viết **gọn** hơn thế nào, và có cách nào **không cần so sánh** (nhanh hơn `O(n log n)`)?

## 6. Cách 1 (chính) — `sorted` với key tuple

```python
return sorted(arr, key=lambda x: (bin(x).count("1"), x))
```

Đọc từng phần:
- `bin(x).count("1")` — tiêu chí chính: số bit 1.
- `x` — tiêu chí phụ khi hòa: giá trị.
- `sorted(...)` — sắp tăng dần theo tuple khóa.

In [12]:
def sort_by_bits_key(arr):
    return sorted(arr, key=lambda x: (bin(x).count("1"), x))

print(sort_by_bits_key(arr1))
print(sort_by_bits_key(arr2))

[0, 1, 2, 4, 8, 3, 5, 6, 7]
[1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]


## 7. TRACE — nhìn thấy khóa của từng phần tử

Với `arr = [7, 3, 8, 5, 0, 6, 4]`:

| x | nhị phân | số bit | khóa |
|---|---|---|---|
| 7 | 111 | 3 | (3, 7) |
| 3 | 11 | 2 | (2, 3) |
| 8 | 1000 | 1 | (1, 8) |
| 5 | 101 | 2 | (2, 5) |
| 0 | 0 | 0 | (0, 0) |
| 6 | 110 | 2 | (2, 6) |
| 4 | 100 | 1 | (1, 4) |

Sắp khóa: `(0,0) < (1,4) < (1,8) < (2,3) < (2,5) < (2,6) < (3,7)` → `[0, 4, 8, 3, 5, 6, 7]`.

In [13]:
def trace_sort_by_bits(arr):
    print(f"{'x':>5} | {'nhị phân':>10} | bit | khóa")
    keyed = []
    for x in arr:
        k = (bin(x).count("1"), x)
        keyed.append(k)
        print(f"{x:>5} | {bin(x)[2:]:>10} | {k[0]:>3} | {k}")
    keyed.sort()
    print("Khóa sau khi sắp:", keyed)
    result = [x for _, x in keyed]
    print("Kết quả:", result)
    return result

trace_sort_by_bits([7, 3, 8, 5, 0, 6, 4])

    x |   nhị phân | bit | khóa
    7 |        111 |   3 | (3, 7)
    3 |         11 |   2 | (2, 3)
    8 |       1000 |   1 | (1, 8)
    5 |        101 |   2 | (2, 5)
    0 |          0 |   0 | (0, 0)
    6 |        110 |   2 | (2, 6)
    4 |        100 |   1 | (1, 4)
Khóa sau khi sắp: [(0, 0), (1, 4), (1, 8), (2, 3), (2, 5), (2, 6), (3, 7)]
Kết quả: [0, 4, 8, 3, 5, 6, 7]


[0, 4, 8, 3, 5, 6, 7]

### 7.1. Trace nhóm theo số bit

Một cách nhìn khác: gom các số theo số bit, mỗi nhóm sắp tăng dần, rồi nối nhóm 0, 1, 2, … Đây cũng là ý tưởng của cách bucket ở mục 10.

In [14]:
from collections import defaultdict

def show_groups(arr):
    groups = defaultdict(list)
    for x in arr:
        groups[bin(x).count("1")].append(x)
    for bits in sorted(groups):
        print(f"{bits} bit: {sorted(groups[bits])}")

show_groups([0, 1, 2, 3, 4, 5, 6, 7, 8])

0 bit: [0]
1 bit: [1, 2, 4, 8]
2 bit: [3, 5, 6]
3 bit: [7]


`defaultdict(list)` là dict mà khi truy cập key **chưa có**, nó tự tạo giá trị mặc định `list()` = `[]`. Nhờ vậy viết `groups[k].append(x)` mà không cần kiểm tra `k in groups`.

In [15]:
d = defaultdict(list)
d["a"].append(1)
d["a"].append(2)
d["b"].append(3)
print(dict(d))

{'a': [1, 2], 'b': [3]}


## 8. Cách 2 — Comparator với `functools.cmp_to_key`

Ở nhiều ngôn ngữ (C++, Java) người ta viết hàm so sánh `cmp(a, b)` trả **số âm** nếu `a` đứng trước, **dương** nếu `b` đứng trước, `0` nếu ngang nhau. Python 3 không nhận `cmp` trực tiếp, phải bọc bằng `cmp_to_key`.

In [16]:
from functools import cmp_to_key

def compare(a, b):
    ca, cb = bin(a).count("1"), bin(b).count("1")
    if ca != cb:
        return ca - cb      # ít bit hơn đứng trước
    return a - b            # hòa: nhỏ hơn đứng trước

def sort_by_bits_cmp(arr):
    return sorted(arr, key=cmp_to_key(compare))

print(compare(3, 4), compare(4, 3), compare(5, 6))
print(sort_by_bits_cmp(arr1))

1 -1 -1
[0, 1, 2, 4, 8, 3, 5, 6, 7]


Comparator **linh hoạt** (viết được quy tắc kỳ lạ) nhưng chậm hơn key vì mỗi lần so sánh lại phải tính popcount (khoảng `n log n` lần thay vì `n` lần). Với key, Python tính khóa **một lần** cho mỗi phần tử.

## 9. Cách 3 — Mã hoá khóa thành **một số nguyên**

Vì `0 <= x <= 10^4`, ta có thể gói hai tiêu chí vào một số: `key = bits * 10001 + x`. Tiêu chí chính nhân với hệ số lớn hơn mọi giá trị của tiêu chí phụ → so sánh `key` giống so sánh tuple.

Giải thích: nếu `bits_a < bits_b` thì `bits_a*10001 + x_a <= bits_a*10001 + 10000 < (bits_a+1)*10001 <= bits_b*10001 + x_b`.

Cũng có thể dùng dịch bit: `(bits << 14) | x` vì `10^4 < 2^14 = 16384`.

In [17]:
def sort_by_bits_encoded(arr):
    return sorted(arr, key=lambda x: bin(x).count("1") * 10001 + x)

def sort_by_bits_shift(arr):
    return sorted(arr, key=lambda x: (bin(x).count("1") << 14) | x)

print(sort_by_bits_encoded(arr1))
print(sort_by_bits_shift(arr1))
print("2^14 =", 1 << 14)

[0, 1, 2, 4, 8, 3, 5, 6, 7]
[0, 1, 2, 4, 8, 3, 5, 6, 7]
2^14 = 16384


Mẹo này hữu ích ở ngôn ngữ không có tuple, hoặc khi muốn sắp **tại chỗ** một mảng số nguyên rồi "giải mã" bằng `key % 10001`. Nhược điểm: phụ thuộc vào **giới hạn đề** — đổi constraint là phải đổi hệ số.

## 10. Cách 4 — Bucket / Counting theo số bit (không cần sắp theo bit)

Số bit của `x <= 10^4` tối đa là 13 (`8191 = 1111111111111₂`, còn `10^4 < 16384 = 2^14`). Ý tưởng:
1. Sắp `arr` **một lần** theo giá trị.
2. Đi qua mảng đã sắp, bỏ mỗi số vào **xô** (bucket) đúng số bit của nó — thứ tự trong xô tự động tăng dần.
3. Nối xô 0, 1, 2, …, 14.

Nếu thay bước 1 bằng **counting sort** theo giá trị (vì giá trị ≤ 10^4), toàn bộ là `O(n + V)` với `V = 10^4`.

In [18]:
def sort_by_bits_bucket(arr):
    buckets = [[] for _ in range(15)]      # số bit từ 0 đến 14
    for x in sorted(arr):
        buckets[bin(x).count("1")].append(x)
    result = []
    for bucket in buckets:
        result.extend(bucket)
    return result

print(sort_by_bits_bucket(arr1))
print(sort_by_bits_bucket(arr2))
print("Số bit lớn nhất với x ≤ 10^4:", max(bin(x).count("1") for x in range(10001)))

[0, 1, 2, 4, 8, 3, 5, 6, 7]
[1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
Số bit lớn nhất với x ≤ 10^4: 13


Chú ý `[[] for _ in range(15)]` tạo **15 list riêng biệt**. Đừng viết `[[]] * 15` — đó là 15 tham chiếu tới **cùng một** list (xem lỗi 3 ở mục 13).

`list.extend(other)` nối mọi phần tử của `other` vào cuối list (khác `append` thêm cả list như một phần tử).

In [19]:
r = [1, 2]
r.extend([3, 4])
print(r)
r.append([5, 6])
print(r)

[1, 2, 3, 4]
[1, 2, 3, 4, [5, 6]]


### 10.1. Bản counting sort thuần (không dùng `sorted`)

Đếm tần suất từng giá trị `0..10^4` bằng mảng `freq`, rồi duyệt giá trị **tăng dần** để đổ vào xô. Không có phép so sánh nào.

In [20]:
def sort_by_bits_counting(arr):
    MAXV = 10_000
    freq = [0] * (MAXV + 1)
    for x in arr:
        freq[x] += 1
    buckets = [[] for _ in range(15)]
    for v in range(MAXV + 1):
        if freq[v]:
            buckets[bin(v).count("1")].extend([v] * freq[v])
    return [x for bucket in buckets for x in bucket]

print(sort_by_bits_counting([3, 3, 0, 7, 1, 1]))

[0, 1, 1, 3, 3, 7]


`10_000` — dấu gạch dưới chỉ để dễ đọc, bằng `10000`. `[v] * k` tạo list gồm `k` bản sao của `v` (an toàn vì `v` là số, không phải list).

## 11. Code nộp LeetCode

Bản ngắn gọn, phổ biến nhất (Cách 1):

In [21]:
class Solution:
    def sortByBits(self, arr: list[int]) -> list[int]:
        return sorted(arr, key=lambda x: (bin(x).count("1"), x))

sol = Solution()
print(sol.sortByBits([0,1,2,3,4,5,6,7,8]))
print(sol.sortByBits([1024,512,256,128,64,32,16,8,4,2,1]))

[0, 1, 2, 4, 8, 3, 5, 6, 7]
[1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]


Bản bucket (Cách 4) — cũng Accepted:

In [22]:
class SolutionBucket:
    def sortByBits(self, arr: list[int]) -> list[int]:
        buckets = [[] for _ in range(15)]
        for x in sorted(arr):
            buckets[bin(x).count("1")].append(x)
        return [x for bucket in buckets for x in bucket]

print(SolutionBucket().sortByBits([0,1,2,3,4,5,6,7,8]))

[0, 1, 2, 4, 8, 3, 5, 6, 7]


## 12. Vì sao đúng? Độ phức tạp

**Tính đúng của Cách 1:** đề định nghĩa thứ tự: `a` đứng trước `b` khi `bits(a) < bits(b)`, hoặc `bits(a) == bits(b)` và `a < b`. Đây **chính xác** là thứ tự từ điển của tuple `(bits, value)`. `sorted` trả về dãy tăng theo khóa → đúng định nghĩa.

**Tính đúng của Cách 4:** invariant — sau khi duyệt các số đã sắp tăng dần, mỗi xô chứa các số cùng số bit **theo thứ tự tăng**. Nối xô theo số bit tăng dần → tiêu chí chính đúng giữa các xô, tiêu chí phụ đúng trong xô.

Gọi `n = len(arr)`, `B` = số bit tối đa (≤ 14), `V = 10^4`.

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| Baseline cặp | `O(n log n + n·B)` | `O(n)` |
| 1. key tuple | `O(n log n + n·B)` | `O(n)` |
| 2. comparator | `O(n log n · B)` | `O(n)` |
| 3. khóa một số | `O(n log n + n·B)` | `O(n)` |
| 4. bucket (+ sorted) | `O(n log n + n·B)` | `O(n)` |
| 4'. counting thuần | `O(n + V·B)` | `O(V + n)` |

Vì `B` là hằng nhỏ (≤ 14), thường viết gọn `O(n log n)`.

### 12.1. Bộ test: mẫu, biên, phản ví dụ, đối chiếu

In [23]:
import random

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.sortByBits([0,1,2,3,4,5,6,7,8]), [0,1,2,4,8,3,5,6,7])
check("đề 2", sol.sortByBits([1024,512,256,128,64,32,16,8,4,2,1]), [1,2,4,8,16,32,64,128,256,512,1024])
check("một phần tử", sol.sortByBits([5]), [5])
check("chỉ số 0", sol.sortByBits([0, 0]), [0, 0])
check("trùng lặp", sol.sortByBits([3, 1, 3, 1]), [1, 1, 3, 3])
check("biên 10^4", sol.sortByBits([10000, 8191, 8192]), [8192, 10000, 8191])
check("hòa nhiều", sol.sortByBits([6, 5, 3, 9, 10, 12]), [3, 5, 6, 9, 10, 12])

PASS đề 1: [0, 1, 2, 4, 8, 3, 5, 6, 7]
PASS đề 2: [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
PASS một phần tử: [5]
PASS chỉ số 0: [0, 0]
PASS trùng lặp: [1, 1, 3, 3]
PASS biên 10^4: [8192, 10000, 8191]
PASS hòa nhiều: [3, 5, 6, 9, 10, 12]


In [24]:
methods = [sort_by_bits_pairs, sort_by_bits_key, sort_by_bits_cmp, sort_by_bits_encoded,
           sort_by_bits_shift, sort_by_bits_bucket, sort_by_bits_counting,
           Solution().sortByBits, SolutionBucket().sortByBits]
rng = random.Random(1356)
for _ in range(300):
    a = [rng.randint(0, 10_000) for _ in range(rng.randint(1, 60))]
    expected = sort_by_bits_pairs(a)
    for f in methods:
        assert f(list(a)) == expected, f
print(f"300 mảng ngẫu nhiên × {len(methods)} cách: khớp nhau — PASS")

300 mảng ngẫu nhiên × 9 cách: khớp nhau — PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên tiêu chí phụ
```python
sorted(arr, key=lambda x: bin(x).count("1"))   # chỉ một tiêu chí
```
Vì `sorted` stable, các số hòa giữ **thứ tự ban đầu**, không phải tăng dần → sai khi input chưa sắp.

In [25]:
a = [8, 4, 2, 1]
print("Sai :", sorted(a, key=lambda x: bin(x).count("1")))
print("Đúng:", sorted(a, key=lambda x: (bin(x).count("1"), x)))

Sai : [8, 4, 2, 1]
Đúng: [1, 2, 4, 8]


### Lỗi 2: Đếm `len(bin(x))` thay vì đếm `'1'`
`len(bin(x))` là **độ dài** chuỗi (kể cả `'0b'`), không phải số bit 1.

In [26]:
x = 8
print("len(bin(8)) =", len(bin(x)), "| số bit 1 =", bin(x).count("1"))

len(bin(8)) = 6 | số bit 1 = 1


### Lỗi 3: Tạo bucket bằng `[[]] * k`
Mọi phần tử là **cùng một list** → thêm vào một xô là thêm vào tất cả.

In [27]:
wrong = [[]] * 3
wrong[0].append(7)
print("Sai :", wrong)
right = [[] for _ in range(3)]
right[0].append(7)
print("Đúng:", right)

Sai : [[7], [7], [7]]
Đúng: [[7], [], []]


### Lỗi 4: Dùng `arr.sort()` rồi `return arr.sort()`
`list.sort()` trả `None`. Hoặc `arr.sort(key=...)` rồi `return arr`, hoặc `return sorted(arr, key=...)`.

In [28]:
def wrong_return(arr):
    return arr.sort(key=lambda x: (bin(x).count("1"), x))

def right_return(arr):
    arr.sort(key=lambda x: (bin(x).count("1"), x))
    return arr

print("Sai :", wrong_return([3, 1, 2]))
print("Đúng:", right_return([3, 1, 2]))

Sai : None
Đúng: [1, 2, 3]


### Lỗi 5: Dùng hệ số mã hoá quá nhỏ
`bits * 100 + x` sai vì `x` có thể tới `10^4 > 100`: số 1 bit lớn (1024) bị xếp sau số 2 bit nhỏ (3).

In [29]:
a = [1024, 3]
print("Sai :", sorted(a, key=lambda x: bin(x).count("1") * 100 + x))
print("Đúng:", sorted(a, key=lambda x: bin(x).count("1") * 10001 + x))

Sai : [3, 1024]
Đúng: [1024, 3]


### Mẹo nhận dạng pattern

- Đề có cụm **"sort by X, and in case of tie sort by Y"** → `sorted(..., key=lambda v: (X(v), Y(v)))`.
- Muốn một tiêu chí **giảm dần** với số: đổi dấu `-X(v)`. Với chuỗi không đổi dấu được → dùng hai lần sort ổn định hoặc `cmp_to_key`.
- Khóa có miền giá trị **nhỏ** (số bit ≤ 14, tần suất ≤ n, chữ cái 26) → nghĩ tới **bucket/counting** để bỏ bước so sánh.
- Đếm bit: `bin(x).count('1')`, `x.bit_count()`, hoặc `x &= x - 1`.
- Bài anh em: *Number of 1 Bits* (191), *Counting Bits* (338), *Sort Array by Increasing Frequency* (1636), *Sort Characters By Frequency* (451).

## 14. Biến thể

### 14.1. Số bit **giảm dần**, hòa thì giá trị **tăng dần**
Chỉ cần đổi dấu tiêu chí chính.

In [30]:
def sort_bits_desc(arr):
    return sorted(arr, key=lambda x: (-bin(x).count("1"), x))

print(sort_bits_desc([0, 1, 2, 3, 4, 5, 6, 7, 8]))

[7, 3, 5, 6, 1, 2, 4, 8, 0]


### 14.2. Sắp theo tần suất (LeetCode 1636)
Tần suất tăng dần; hòa thì **giá trị giảm dần**. `Counter` (trong `collections`) đếm tần suất: `Counter([1,1,2])` → `{1: 2, 2: 1}`.

In [31]:
from collections import Counter

def frequency_sort(nums):
    freq = Counter(nums)
    return sorted(nums, key=lambda x: (freq[x], -x))

print(Counter([1, 1, 2, 2, 2, 3]))
print(frequency_sort([1, 1, 2, 2, 2, 3]))      # [3, 1, 1, 2, 2, 2]
print(frequency_sort([2, 3, 1, 3, 2]))         # [1, 3, 3, 2, 2]

Counter({2: 3, 1: 2, 3: 1})
[3, 1, 1, 2, 2, 2]
[1, 3, 3, 2, 2]


### 14.3. Gom nhóm theo số bit (trả dict)

In [32]:
def group_by_bits(arr):
    groups = defaultdict(list)
    for x in sorted(arr):
        groups[bin(x).count("1")].append(x)
    return dict(groups)

print(group_by_bits([0, 1, 2, 3, 4, 5, 6, 7, 8]))

{0: [0], 1: [1, 2, 4, 8], 2: [3, 5, 6], 3: [7]}


## 15. Bài tập (TODO — đáp án ở cuối)

### Bài 1 — Dự đoán output
Không chạy code, hãy viết kết quả của `sortByBits([10, 100, 1000, 10000])`. (Gợi ý: `10 = 1010₂`, `100 = 1100100₂`, `1000 = 1111101000₂`, `10000 = 10011100010000₂`.)

### Bài 2 — Popcount thủ công
Viết `exercise_popcount(x)` **không** dùng `bin` hay `bit_count`.

In [33]:
def exercise_popcount(x):
    # TODO: dùng x & 1 và x >>= 1, hoặc x &= x - 1
    pass

# Gợi ý test:
# assert exercise_popcount(0) == 0
# assert exercise_popcount(7) == 3
# assert exercise_popcount(10000) == 5

### Bài 3 — Hoàn thiện sortByBits với key tuple

In [34]:
def exercise_sort_by_bits(arr):
    # TODO: return sorted(arr, key=...)
    pass

# Gợi ý test:
# assert exercise_sort_by_bits([0,1,2,3,4,5,6,7,8]) == [0,1,2,4,8,3,5,6,7]

### Bài 4 — Viết bằng hai lần sort ổn định
Không dùng tuple; dùng hai lần `sorted`/`sort` với **một** tiêu chí mỗi lần.

In [35]:
def exercise_sort_two_pass(arr):
    # TODO: sắp theo tiêu chí phụ trước, rồi tiêu chí chính
    pass

# Gợi ý test:
# assert exercise_sort_two_pass([1024, 3, 512, 5]) == [512, 1024, 3, 5]

### Bài 5 — Biến thể: sắp chuỗi theo số nguyên âm
Sắp list từ theo **số nguyên âm** (`a e i o u`) tăng dần; hòa thì theo **độ dài giảm dần**; vẫn hòa thì theo **bảng chữ cái**.

In [36]:
def exercise_sort_words(words):
    # TODO: key là tuple 3 thành phần
    pass

# Gợi ý test:
# assert exercise_sort_words(["tree", "sky", "apple", "aa", "bob"]) == ["sky", "bob", "apple", "tree", "aa"]

### Bài 6 — Mini challenge: bucket không dùng `sorted`
Viết lại Cách 4 bằng counting sort thuần (giá trị trong `[0, 10^4]`), trả về đúng thứ tự đề.

In [37]:
def exercise_sort_by_bits_counting(arr):
    # TODO: freq theo giá trị, duyệt v tăng dần, đổ vào bucket theo số bit
    pass

# Gợi ý test:
# assert exercise_sort_by_bits_counting([3, 3, 0, 7, 1, 1]) == [0, 1, 1, 3, 3, 7]

## 16. Gợi ý 3 mức

**Bài 1** — (1) Đếm số chữ số 1 trong từng chuỗi nhị phân. (2) Số bit: 10→2, 100→3, 1000→6, 10000→5. (3) Sắp theo số bit.

**Bài 2** — (1) Mỗi vòng xử lý một bit. (2) `count += x & 1` rồi `x >>= 1`. (3) `while x: x &= x - 1; count += 1`.

**Bài 3** — (1) Khóa hai tiêu chí. (2) `lambda x: (số_bit, x)`. (3) `sorted(arr, key=lambda x: (bin(x).count('1'), x))`.

**Bài 4** — (1) Stable sort giữ thứ tự khi hòa. (2) Lần 1: `sorted(arr)`; lần 2: theo số bit. (3) `return sorted(sorted(arr), key=lambda x: bin(x).count('1'))`.

**Bài 5** — (1) Độ dài giảm dần → `-len(w)`. (2) Đếm nguyên âm: `sum(ch in 'aeiou' for ch in w)`. (3) `key=lambda w: (vowels(w), -len(w), w)`.

**Bài 6** — (1) `freq = [0] * 10001`. (2) Duyệt `v` từ 0 lên, `buckets[bits(v)].extend([v] * freq[v])`. (3) Nối các bucket.

## 17. Đáp án tham khảo

**Bài 1:** số bit 10→2, 100→3, 10000→5, 1000→6 → `[10, 100, 10000, 1000]`.

In [38]:
assert Solution().sortByBits([10, 100, 1000, 10000]) == [10, 100, 10000, 1000]

def answer_popcount(x):
    count = 0
    while x:
        x &= x - 1
        count += 1
    return count

def answer_sort_by_bits(arr):
    return sorted(arr, key=lambda x: (bin(x).count("1"), x))

def answer_sort_two_pass(arr):
    return sorted(sorted(arr), key=lambda x: bin(x).count("1"))

def answer_sort_words(words):
    def vowels(w):
        return sum(ch in "aeiou" for ch in w)
    return sorted(words, key=lambda w: (vowels(w), -len(w), w))

def answer_sort_by_bits_counting(arr):
    freq = [0] * 10001
    for x in arr:
        freq[x] += 1
    buckets = [[] for _ in range(15)]
    for v in range(10001):
        if freq[v]:
            buckets[answer_popcount(v)].extend([v] * freq[v])
    return [x for bk in buckets for x in bk]

assert answer_popcount(0) == 0 and answer_popcount(7) == 3 and answer_popcount(10000) == 5
assert answer_sort_by_bits([0,1,2,3,4,5,6,7,8]) == [0,1,2,4,8,3,5,6,7]
assert answer_sort_two_pass([1024, 3, 512, 5]) == [512, 1024, 3, 5]
assert answer_sort_words(["tree", "sky", "apple", "aa", "bob"]) == ["sky", "bob", "apple", "tree", "aa"]
assert answer_sort_by_bits_counting([3, 3, 0, 7, 1, 1]) == [0, 1, 1, 3, 3, 7]

rng = random.Random(99)
for _ in range(200):
    a = [rng.randint(0, 10_000) for _ in range(rng.randint(1, 40))]
    e = sort_by_bits_key(a)
    assert answer_sort_by_bits(a) == answer_sort_two_pass(a) == answer_sort_by_bits_counting(a) == e
print("Đáp án bài tập: PASS toàn bộ")

Đáp án bài tập: PASS toàn bộ


## 18. Tổng kết — template nhớ nhanh + checklist

```python
# Sắp theo nhiều tiêu chí
sorted(items, key=lambda v: (tiêu_chí_1(v), tiêu_chí_2(v), ...))
# giảm dần với số: dùng -tiêu_chí(v)

# Popcount
bin(x).count("1")      # hoặc x.bit_count() (Python 3.10+)
while x: x &= x - 1; c += 1
```

**Checklist trước khi submit**
- [ ] Khóa có **đủ hai** tiêu chí: số bit, rồi giá trị.
- [ ] Đếm `'1'`, không dùng `len(bin(x))`.
- [ ] Trả về list (không `return arr.sort()`).
- [ ] Bucket tạo bằng list comprehension, không `[[]] * k`.
- [ ] Đã test số `0`, phần tử trùng, các số cùng số bit ở thứ tự lộn xộn.